# Inventing Reinforcement Learning

You won't read about RL — you'll play a game, get frustrated, build strategies, and accidentally invent every major RL algorithm along the way.


## Part 1: Let's Play a Game

Forget theory. We're going to play a game. The game is called **CartPole** — there's a cart on a track with a pole balanced on top. You can push the cart left or right. The goal: keep the pole from falling over as long as possible.

The twist? You play using Python code.


<div class="cartpole-demo">
  <svg viewBox="0 0 400 180" xmlns="http://www.w3.org/2000/svg">
    <!-- track -->
    <line x1="20" y1="155" x2="380" y2="155" stroke="#888" stroke-width="2"/>
    <line x1="20" y1="155" x2="20" y2="148" stroke="#888" stroke-width="2"/>
    <line x1="380" y1="155" x2="380" y2="148" stroke="#888" stroke-width="2"/>
    <!-- cart -->
    <rect x="160" y="130" width="80" height="30" rx="4" fill="#c8a96e" stroke="#7a6b3a" stroke-width="1.5"/>
    <!-- wheels -->
    <circle cx="178" cy="160" r="6" fill="#555"/>
    <circle cx="222" cy="160" r="6" fill="#555"/>
    <!-- pole (animated) -->
    <g class="cartpole-pole">
      <line x1="200" y1="130" x2="200" y2="40" stroke="#333" stroke-width="5" stroke-linecap="round"/>
      <circle cx="200" cy="40" r="6" fill="#c0392b"/>
    </g>
    <!-- pivot -->
    <circle cx="200" cy="130" r="4" fill="#333"/>
    <!-- arrows -->
    <text x="120" y="150" font-size="18" fill="#888" font-family="system-ui">&larr;</text>
    <text x="268" y="150" font-size="18" fill="#888" font-family="system-ui">&rarr;</text>
  </svg>
  <p class="cartpole-caption">The CartPole environment: keep the pole balanced by pushing the cart left or right.</p>
</div>


### Exercise 1.1 — Set Up the Game

First, install the game environment and create a CartPole game. Run `env.reset()` — it gives you back 4 numbers. Print them. What do you think they mean?


In [ ]:
# Run this first (only needed once)
# !pip install gymnasium

import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=42)
print("Observation:", obs)
print("Number of values:", len(obs))

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

You should see 4 numbers. They describe the current state of the cart and pole — position, speed, angle, and how fast the angle is changing.
</details>

<details><summary>Solution</summary>

The 4 numbers are:
- `obs[0]`: cart position (negative = left, positive = right)
- `obs[1]`: cart velocity
- `obs[2]`: pole angle (negative = tilting left, positive = tilting right)
- `obs[3]`: pole angular velocity (how fast the angle is changing)


</details>

The 4 numbers describe the world:

| Index | What it is | Meaning |
|-------|-----------|---------|
| `obs[0]` | Cart position | Where the cart is on the track |
| `obs[1]` | Cart velocity | How fast the cart is moving |
| `obs[2]` | Pole angle | Which way the pole is leaning |
| `obs[3]` | Angular velocity | How fast the pole is falling |

You have two possible actions: **0 = push left**, **1 = push right**.


### Exercise 1.2 — Take a Step

Call `env.step(action)` with action 0 (push left) or 1 (push right). It returns 5 things. Print them all and figure out what each one means.


In [ ]:
obs, info = env.reset(seed=42)
result = env.step(1)  # push right
print("Result has", len(result), "items:")
for i, item in enumerate(result):
    print(f"  [{i}]: {item}")

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

The 5 return values are: (new_observation, reward, terminated, truncated, info). The reward is 1.0 for every step you survive. The game ends when `terminated` or `truncated` becomes True.
</details>

<details><summary>Solution</summary>

```python
obs, info = env.reset(seed=42)
new_obs, reward, terminated, truncated, info = env.step(1)
print(f"New observation: {new_obs}")
print(f"Reward: {reward}")          # 1.0 — you survived one more step!
print(f"Terminated: {terminated}")  # False — pole hasn't fallen yet
print(f"Truncated: {truncated}")    # False — haven't hit 500 steps
```

</details>

### Exercise 1.3 — Play a Full Game (Randomly)

Write a loop that keeps taking random actions (0 or 1) until the game ends (`terminated` or `truncated`). Count how many steps you survived — that's your score. Run it a few times. What scores do you get?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Use `np.random.choice([0, 1])` to pick a random action each step.
</details>

<details><summary>Hint 2</summary>

```python
obs, info = env.reset()
total_reward = 0
done = False
while not done:
    action = np.random.choice([0, 1])
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    done = terminated or truncated
print(f"Score: {total_reward}")
```

</details>

<details><summary>Solution</summary>

```python
scores = []
for game in range(10):
    obs, info = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = np.random.choice([0, 1])
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
    scores.append(total_reward)
    print(f"Game {game+1}: score = {total_reward}")
print(f"\nAverage score: {np.mean(scores):.1f}")
```

</details>

### Exercise 1.4 — Develop Your Intuition

Before writing any strategy, just think:


**Your answer:** The pole is tilting to the RIGHT. Should you push the cart left or right to keep it balanced?

**Your answer:** The pole is tilting to the LEFT. Which way should you push?

**Your answer:** The pole is nearly vertical but moving (rotating) fast to the right. What should you do?

<details><summary>Hint 1</summary>

Think about balancing a broomstick on your palm. If it tilts right, you move your hand RIGHT — under the falling point. Same idea here.
</details>

<details><summary>Solution</summary>

If the pole tilts right, push right — move the cart under the pole. If it tilts left, push left. If it's moving fast, you need to act in the direction it's falling. This is the core intuition behind a good policy.


</details>

## Part 2: Build Your Own Player

Random actions give terrible scores. You already have intuition about what to do — let's turn it into code.


### Exercise 2.0 — Helper Functions

Run these helper functions first. They'll save you from rewriting the game loop every time.


In [ ]:
def play_game(policy_fn, env):
    """Play one episode using policy_fn and return total reward."""
    obs, info = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
    return total_reward

def evaluate(policy_fn, env, n=100):
    """Average score over n episodes."""
    scores = [play_game(policy_fn, env) for _ in range(n)]
    return np.mean(scores)

In [ ]:
# YOUR CODE HERE


<details><summary>Solution</summary>


</details>

### Exercise 2.1 — Your First Policy

Write a function `my_policy(obs)` that takes the 4 observation numbers and returns 0 (left) or 1 (right).

Use your intuition from the previous exercise: if the pole is tilting right (positive angle = `obs[2] > 0`), push right (return 1). Otherwise push left.

Evaluate it over 100 episodes. How much better is it than random?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

It's as simple as: `return 1 if obs[2] > 0 else 0`. The angle is `obs[2]`.
</details>

<details><summary>Solution</summary>

```python
def my_policy(obs):
    return 1 if obs[2] > 0 else 0

avg = evaluate(my_policy, env)
print(f"Average score: {avg:.1f}")
# Typically 40-60 — much better than random (~20), but still not great
```

</details>

### Exercise 2.2 — Improve Your Policy

Your angle-only policy probably scores around 40–60. Can you do better?

Try using the angular velocity (`obs[3]`) too. If the pole is tilting right AND the angular velocity is positive (falling faster to the right), it's more urgent to push right. Experiment with different rules. Can you consistently score above 200?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Try: `return 1 if obs[2] + obs[3] > 0 else 0`. Adding the angular velocity helps you anticipate where the pole is GOING, not just where it IS.
</details>

<details><summary>Hint 2</summary>

Try giving angular velocity more weight: `return 1 if obs[2] + 0.5 * obs[3] > 0 else 0`. Or less: `return 1 if obs[2] + 2 * obs[3] > 0 else 0`. Which ratio works best?
</details>

<details><summary>Solution</summary>

```python
def better_policy(obs):
    return 1 if obs[2] + 0.5 * obs[3] > 0 else 0

avg = evaluate(better_policy, env)
print(f"Average score: {avg:.1f}")
# Typically 60-200+ depending on the exact weights
```

</details>

### The Best You Can Do By Hand

Figure out all possible ways you can write the policy function to maximize the score. Try at least 3 different heuristics and record their average scores.

Some ideas to explore:
- Use all 4 observations, not just angle and angular velocity
- Try different combinations and multipliers
- Try non-linear rules (if-else chains, thresholds)

What's the highest average score you can achieve by hand-tuning?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Using all 4 observations with the right multipliers can get you close to 500 (the maximum). But finding those multipliers by hand is tedious...
</details>

## Part 3: What Just Happened?

Let's put names on what you just did:

| What you experienced | What it's called |
|---------------------|-----------------|
| The game (CartPole) | **Environment** |
| The 4 numbers you see each step | **Observations** (or **state**) |
| Push left / push right | **Actions** |
| Number of steps survived | **Reward** |
| Your function (`my_policy`) | A **policy** |
| Play game → see result → tweak policy → repeat | **Reinforcement Learning** |

This is the closest thing to how humans learn. You don't get a textbook — you try stuff, get feedback, and adapt. A child learning to walk, a dog learning tricks, you learning to drive — all reinforcement learning. **That's life!**


### Exercise 3.1 — RL Is Everywhere

Map the RL terms to a real-life example: a child learning to ride a bicycle.


**Your answer:** What is the environment?

**Your answer:** What are the observations (state)?

**Your answer:** What are the actions?

**Your answer:** What are the rewards and penalties?

**Your answer:** What is the policy?

<details><summary>Hint 1</summary>

Environment = the physical world (road, bicycle, gravity). Observations = what the child sees and feels (balance, speed, slope). Actions = turn handlebars, pedal, lean. Rewards = staying upright, moving forward. Penalties = falling, crashing.
</details>

<details><summary>Solution</summary>

- Environment: the physical world — road, bike, gravity, wind
- Observations: sense of balance, speed, visual input, slope
- Actions: steer, pedal, brake, lean left/right
- Rewards: staying upright, reaching destination. Penalties: falling, crashing
- Policy: the child's internal rules that map "I'm tilting left" to "lean right"


</details>

## Part 4: Policy as a Formula

Look at your best heuristic from Part 2. It probably looked something like: "if angle > 0, go right" or "if angle + 0.5 × angular_velocity > 0, go right." That's really just: **multiply each observation by some number, add them up, and check the sign**. Your action is a function of the observations.


### Exercise 4.1 — Two-Weight Policy

Write a policy that computes `score = w1 * obs[2] + w2 * obs[3]` (angle and angular velocity). If score > 0, go right. Otherwise go left.

Try these weight combinations and record the average score for each:
- w1=1, w2=0
- w1=1, w2=1
- w1=1, w2=0.5
- w1=0.5, w2=1


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

```python
def two_weight_policy(obs, w1, w2):
    return 1 if w1 * obs[2] + w2 * obs[3] > 0 else 0

for w1, w2 in [(1,0), (1,1), (1,0.5), (0.5,1)]:
    policy = lambda obs, w1=w1, w2=w2: two_weight_policy(obs, w1, w2)
    print(f"w1={w1}, w2={w2}: avg score = {evaluate(policy, env):.1f}")
```

</details>

<details><summary>Solution</summary>

```python
def two_weight_policy(obs, w1, w2):
    return 1 if w1 * obs[2] + w2 * obs[3] > 0 else 0

for w1, w2 in [(1,0), (1,1), (1,0.5), (0.5,1)]:
    policy = lambda obs, w1=w1, w2=w2: two_weight_policy(obs, w1, w2)
    print(f"w1={w1}, w2={w2}: avg score = {evaluate(policy, env):.1f}")
```

</details>

### Exercise 4.2 — Four-Weight Policy

Now use ALL 4 observations. Write `weighted_policy(obs, weights)` that computes:

`weights[0]*obs[0] + weights[1]*obs[1] + weights[2]*obs[2] + weights[3]*obs[3]`

Go right if positive, left otherwise. This is a **dot product** — `weights @ obs` in numpy.

Try weights = `[0, 0, 1, 0.5]` (ignoring cart position and velocity). Then try `[0.1, 0.5, 1, 0.5]`.


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

`np.dot(weights, obs)` or `sum(w*o for w,o in zip(weights, obs))` both work.
</details>

<details><summary>Solution</summary>

```python
def weighted_policy(obs, weights):
    return 1 if np.dot(weights, obs) > 0 else 0

for w in [[0, 0, 1, 0.5], [0.1, 0.5, 1, 0.5]]:
    policy = lambda obs, w=w: weighted_policy(obs, np.array(w))
    print(f"weights={w}: avg score = {evaluate(policy, env):.1f}")
```

</details>

The numbers you've been manually tuning — w1, w2, w3, w4 — are called **policy parameters**. The function that maps observations to actions is the **policy**. The entire goal of reinforcement learning just became a concrete optimization problem:

**Find the weights (policy parameters) that maximize the average reward.**

But how? Trying them by hand is tedious. We need a strategy.


## Part 5: The Search for Best Parameters

You need 4 good numbers. How hard can it be?


### Exercise 5.1 — Brute Force?

One idea: try ALL possible combinations of the 4 weights.


**Your answer:** Each weight is a real number (like 0.37 or -1.2). How many possible values does a single weight have?

**Your answer:** If you discretized each weight to 100 values (from -1 to 1 in steps of 0.02), how many combinations of 4 weights would you need to try?

**Your answer:** Why is this approach impractical?

<details><summary>Hint 1</summary>

100 × 100 × 100 × 100 = 100 million combinations. And each one requires playing multiple games to get a reliable score. That would take days.
</details>

<details><summary>Solution</summary>

Each weight is continuous — infinitely many values. Even discretized to 100 values each, you'd have 100^4 = 100,000,000 combinations. At 5 games per evaluation, that's 500 million games. Brute force doesn't scale.


</details>

### Exercise 5.2 — Your Ideas

Think for a minute. What strategies can YOU think of for finding 4 good weight values without trying every combination? Write down at least 2 ideas before looking at hints.


**Your answer:** Strategy 1?

**Your answer:** Strategy 2?

<details><summary>Hint 1</summary>

One approach: try a bunch of random weights and keep the best one. Another: start with some weights and make small adjustments to improve them.
</details>

<details><summary>Hint 2</summary>

Yet another: take the best weights you found, make random variations of them, keep the best variations, repeat. Sound like anything from biology?
</details>

<details><summary>Solution</summary>


</details>

### Exercise 5.3 — Strategy 1 — Random Search

Generate 100 random weight vectors (each weight between -1 and 1). Evaluate each one over 5 episodes. Print the best weights and their score.


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

`np.random.uniform(-1, 1, size=4)` generates one random weight vector.
</details>

<details><summary>Hint 2</summary>

```python
best_score = 0
best_weights = None
for i in range(100):
    weights = np.random.uniform(-1, 1, size=4)
    policy = lambda obs, w=weights: weighted_policy(obs, w)
    score = evaluate(policy, env, n=5)
    if score > best_score:
        best_score = score
        best_weights = weights
```

</details>

<details><summary>Solution</summary>

```python
best_score = 0
best_weights = None
for i in range(100):
    weights = np.random.uniform(-1, 1, size=4)
    policy = lambda obs, w=weights: weighted_policy(obs, w)
    score = evaluate(policy, env, n=5)
    if score > best_score:
        best_score = score
        best_weights = weights.copy()

print(f"Best score: {best_score:.1f}")
print(f"Best weights: {best_weights}")

# Verify with more episodes
policy = lambda obs, w=best_weights: weighted_policy(obs, w)
print(f"Verified over 100 episodes: {evaluate(policy, env, n=100):.1f}")
```

</details>

### Exercise 5.4 — Strategy 2 — Evolve!

Random search found a decent solution, but can we do better? Here's an idea:

1. Start with 100 random weight vectors
2. Evaluate all of them
3. Keep only the **top 20**

Now you have 20 good weight vectors. But you need 100 for the next round. How would you get from 20 back to 100? And then what? Design the rest of this algorithm yourself, implement it, and run it for 10 generations. Print the best score each generation.


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Think about it: each of the 20 survivors is a good set of weights. What if you made slightly different copies of each one — like variations on a theme? How many copies per survivor do you need to get back to 100?
</details>

<details><summary>Hint 2</summary>

Create 4 'children' from each of the 20 by adding small random noise: `child = parent + np.random.uniform(-0.1, 0.1, size=4)`. That gives 80 children + 20 parents = 100. Then repeat: evaluate → keep top 20 → breed → repeat.
</details>

<details><summary>Hint 3</summary>

```python
population = [np.random.uniform(-1, 1, size=4) for _ in range(100)]

for gen in range(10):
    scored = []
    for weights in population:
        policy = lambda obs, w=weights: weighted_policy(obs, w)
        score = evaluate(policy, env, n=5)
        scored.append((score, weights))

    scored.sort(key=lambda x: -x[0])
    top_20 = [w for _, w in scored[:20]]
    print(f"Gen {gen+1}: best = {scored[0][0]:.1f}")

    population = list(top_20)
    for parent in top_20:
        for _ in range(4):
            child = parent + np.random.uniform(-0.1, 0.1, size=4)
            population.append(child)
```

</details>

<details><summary>Solution</summary>

```python
population = [np.random.uniform(-1, 1, size=4) for _ in range(100)]

for gen in range(10):
    scored = []
    for weights in population:
        policy = lambda obs, w=weights: weighted_policy(obs, w)
        score = evaluate(policy, env, n=5)
        scored.append((score, weights))

    scored.sort(key=lambda x: -x[0])
    top_20 = [w.copy() for _, w in scored[:20]]
    print(f"Gen {gen+1}: best = {scored[0][0]:.1f}, avg top-20 = {np.mean([s for s,_ in scored[:20]]):.1f}")

    population = list(top_20)
    for parent in top_20:
        for _ in range(4):
            child = parent + np.random.uniform(-0.1, 0.1, size=4)
            population.append(child)

# Final best
best_weights = top_20[0]
policy = lambda obs, w=best_weights: weighted_policy(obs, w)
print(f"\nFinal best weights: {best_weights}")
print(f"Verified over 100 episodes: {evaluate(policy, env, n=100):.1f}")
```

</details>

What you just invented is called a **Genetic Algorithm**. See the analogy?

| Your code | Biology |
|-----------|---------|
| Weight vector | DNA |
| Score | Fitness |
| Top 20 survive | Natural selection |
| Adding noise to create children | Mutation |
| Repeat for generations | Evolution |

You found good policy parameters without any calculus — just random variation and selection. Evolution works!


### Improve Evolution

Can you think of ways to make the genetic algorithm better? Try implementing any of these ideas:

- **Crossover:** instead of just mutating one parent, combine two parents (e.g., take weights 0-1 from parent A and weights 2-3 from parent B)
- **Adaptive mutation:** start with large noise (±0.5) and reduce it each generation
- **Elitism:** always keep the single best individual unchanged
- **Larger population or more generations**

Try your ideas and see if they beat the basic version! If you invent something new, we want to hear about it.


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

For crossover: `child = np.where(np.random.random(4) > 0.5, parent_a, parent_b)`
</details>

## Part 6: Can Gradients Help?

Genetic algorithms work but feel brute-force. In the Gradient Descent chapter, you learned a more targeted approach: compute the gradient, move in the direction that improves the objective. Can we do that here?


### Exercise 6.1 — Strategy 3 — Tweak and Observe

In the Gradient Descent chapter, you computed gradients by nudging each parameter slightly and measuring how the output changed (finite differences). Can you apply that same trick here?

Take your best weights from the genetic algorithm. Estimate the gradient of the average score with respect to each weight, update the weights in the uphill direction, and repeat 50 times. Does this converge to a good solution?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

For each weight i: add a small epsilon (e.g. 0.01), play 10 games and record the average score. Subtract epsilon, play 10 games. The gradient for that weight ≈ (score_plus - score_minus) / (2 × epsilon). Then update: `weights += learning_rate * gradient`.
</details>

<details><summary>Hint 2</summary>

```python
weights = best_weights.copy()  # from genetic algorithm
lr = 0.01
eps = 0.01

for step in range(50):
    gradient = np.zeros(4)
    for i in range(4):
        w_plus = weights.copy()
        w_plus[i] += eps
        w_minus = weights.copy()
        w_minus[i] -= eps
        score_plus = evaluate(lambda obs, w=w_plus: weighted_policy(obs, w), env, n=10)
        score_minus = evaluate(lambda obs, w=w_minus: weighted_policy(obs, w), env, n=10)
        gradient[i] = (score_plus - score_minus) / (2 * eps)
    weights += lr * gradient
    score = evaluate(lambda obs, w=weights: weighted_policy(obs, w), env, n=10)
    print(f"Step {step+1}: score = {score:.1f}")
```

</details>

<details><summary>Solution</summary>

```python
weights = np.array([0.0, 0.0, 1.0, 0.5])
lr = 0.01
eps = 0.01

for step in range(50):
    gradient = np.zeros(4)
    for i in range(4):
        w_plus = weights.copy()
        w_plus[i] += eps
        w_minus = weights.copy()
        w_minus[i] -= eps
        score_plus = evaluate(lambda obs, w=w_plus: weighted_policy(obs, w), env, n=10)
        score_minus = evaluate(lambda obs, w=w_minus: weighted_policy(obs, w), env, n=10)
        gradient[i] = (score_plus - score_minus) / (2 * eps)
    weights += lr * gradient
    score = evaluate(lambda obs, w=weights: weighted_policy(obs, w), env, n=10)
    if (step + 1) % 10 == 0:
        print(f"Step {step+1}: score = {score:.1f}, weights = {weights}")
```

</details>

### Exercise 6.2 — Why Didn't That Work Well?

The gradient approach probably gave erratic results — the score jumps around instead of steadily improving. Why?


**Your answer:** If you play 10 games with the EXACT same weights, do you always get the same score? Why not?

**Your answer:** If the score varies by ±30 between runs, and you're trying to detect a difference of 0.01 in one weight, can you reliably measure the gradient?

<details><summary>Hint 1</summary>

The game has randomness (different starting states). Your 'gradient' is the difference between two noisy measurements — it's almost pure noise.
</details>

<details><summary>Solution</summary>

The game is stochastic — the same weights give different scores each run. When you change a weight by 0.01, the score difference is swamped by random noise. Your "gradient" is mostly measuring randomness, not the true effect of the weight change.


</details>

There's an even deeper problem. When you push right at step 5, was it the right move? You won't know until the game ends — maybe at step 200. The total score doesn't tell you WHICH of your 200 individual actions were good and which were bad. This is called the **credit assignment problem**: how do you assign credit (or blame) to each action?


### Exercise 6.3 — Assigning Credit

You played a game and scored 150 steps. At step 100, you pushed right.


**Your answer:** Was pushing right at step 100 a good action? How would you know?

**Your answer:** Which reward is more relevant to the action at step 100 — the reward at step 101 or the reward at step 150?

**Your answer:** If you got a reward of 1 at each step, how would you assign 'credit' to the action at step 100? Should nearby rewards count more than distant ones?

<details><summary>Hint 1</summary>

The reward at step 101 is directly influenced by what you did at step 100. The reward at step 150 is influenced by 50 other actions in between. Nearby rewards should count more.
</details>

<details><summary>Solution</summary>

The action at step 100 most directly affects step 101. Each subsequent step is influenced by many intervening actions. So nearby rewards should count more, and distant rewards should be discounted.


</details>

### Exercise 6.4 — Design Your Own Credit System

You agreed that nearby rewards should count more than distant ones. Now design a formula.

Suppose you played 5 steps and got rewards [1, 1, 1, 1, 1]. You want to compute a "total credit" for the action at step 0.


**Your answer:** If you just sum all future rewards equally, what's the credit for step 0? (1+1+1+1+1 = 5). What's the problem with this?

**Your answer:** What if each future reward was worth a little LESS than the one before it — say 99% as much? Step 0's own reward counts fully (×1), step 1's reward counts ×0.99, step 2's counts ×0.99², etc. What would the credit for step 0 be?

**Your answer:** What should the credit for the LAST step (step 4) be? It has no future rewards.

**Your answer:** What would you call that 0.99 multiplier?

<details><summary>Hint 1</summary>

With the 0.99 multiplier: credit for step 0 = 1 + 0.99×1 + 0.99²×1 + 0.99³×1 + 0.99⁴×1 ≈ 4.90. Credit for step 4 = just 1 (no future). The 0.99 is a 'discount factor' — it makes distant rewards fade away.
</details>

<details><summary>Solution</summary>

Credit for step t = reward[t] + 0.99 × reward[t+1] + 0.99² × reward[t+2] + ...

This is called the **discounted return**. The multiplier (0.99) is called the **discount factor** (gamma). It controls how much you care about distant vs. nearby rewards.


</details>

### Exercise 6.5 — Code Your Credit System

Write `compute_discounted_rewards(rewards, gamma=0.99)` that implements the credit system you just designed. It takes a list of rewards (one per step) and returns the discounted return for each step.


In [ ]:
test_rewards = [1, 1, 1, 1, 1]
result = compute_discounted_rewards(test_rewards, gamma=0.99)
print("Discounted rewards:", result)
# Step 0 should be ~4.90, step 4 should be 1.0

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Think about it from the end. The last step's credit is just its own reward. The second-to-last step's credit is its reward + gamma × (last step's credit). Can you see a pattern working backwards?
</details>

<details><summary>Hint 2</summary>

Start from the last step and work backwards: `running = 0`. For each step t from the end: `running = rewards[t] + gamma * running`. Store `running` as that step's credit.
</details>

<details><summary>Hint 3</summary>

```python
def compute_discounted_rewards(rewards, gamma=0.99):
    discounted = [0] * len(rewards)
    running = 0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        discounted[t] = running
    return discounted
```

</details>

<details><summary>Solution</summary>

```python
def compute_discounted_rewards(rewards, gamma=0.99):
    discounted = [0.0] * len(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        discounted[t] = running
    return discounted
```

</details>

### Exercise 6.6 — Look at the Raw Numbers

Play a game, compute discounted rewards, and look at the values. What do you notice?


In [ ]:
# Play one game with your best policy
obs, info = env.reset()
rewards = []
done = False
while not done:
    action = 1 if obs[2] > 0 else 0
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    done = terminated or truncated

disc = compute_discounted_rewards(rewards)
print(f"Game lasted {len(rewards)} steps")
print(f"First step's credit:  {disc[0]:.1f}")
print(f"Last step's credit:   {disc[-1]:.1f}")
print(f"Range: {min(disc):.1f} to {max(disc):.1f}")

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

The first step's credit is a large number (like 40–60) because it includes all future rewards. The last step is just 1.0. All values are positive — can you see a problem? If ALL actions get positive credit, the model can't tell which ones were GOOD vs. just OK.
</details>

<details><summary>Solution</summary>

All the discounted rewards are positive (ranging from 1 to ~50+). That means every action looks "good." But some were clearly better than others — we need a way to separate above-average actions from below-average ones.


</details>

### Exercise 6.7 — Fix the All-Positive Problem

Every action has positive credit, so the model thinks all actions were good. That's not useful.


**Your answer:** If the average credit is 30, and action A got credit 45 while action B got credit 15, which was better?

**Your answer:** What if you subtracted the average from each value? A would become +15 (above average = good!) and B would become -15 (below average = bad). Does this help?

**Your answer:** After subtracting the mean, some values might be ±50 while others are ±5. Should you also scale them? How?

<details><summary>Hint 1</summary>

Subtract the mean so that above-average actions are positive and below-average are negative. Divide by the standard deviation to keep the scale consistent. This is just standard normalization!
</details>

<details><summary>Solution</summary>

Subtract the mean and divide by the standard deviation. Now roughly half the actions have positive values (better than average) and half have negative (worse than average). This gives a clean signal for learning.


</details>

### Exercise 6.8 — Write normalize()

Write a `normalize(values)` function that subtracts the mean and divides by the standard deviation.


In [ ]:
test = [40, 35, 30, 20, 10, 1]
print("Before:", test)
print("After:", normalize(test))
# After normalization: positive = above average, negative = below average

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Use numpy: `np.mean(values)` and `np.std(values)`. Add a tiny number (1e-8) to the standard deviation to prevent division by zero.
</details>

<details><summary>Solution</summary>

```python
def normalize(values):
    values = np.array(values)
    return (values - np.mean(values)) / (np.std(values) + 1e-8)
```

</details>

## Part 7: Teaching a Model to Play

So far you've been hand-tuning weights or using evolution. But there's something unsatisfying — you're searching OUTSIDE the game. You try weights, play a full game, check the score, try new weights. What if the model could learn FROM each game it plays?


### Exercise 7.0 — Why Probabilities?

Your current policy is deterministic: if `weights @ obs > 0`, always go right. Always. Every single time.


**Your answer:** If the policy always makes the same decision in the same state, can it ever discover that the OTHER action might have been better?

**Your answer:** What if instead of 'always go right when score > 0', the model said 'go right with 70% probability'? Sometimes it would go left — accidentally. What could you learn from those accidents?

**Your answer:** If an accidental action leads to a HIGHER reward than usual, what should the model do next time?

<details><summary>Hint 1</summary>

A deterministic policy can't explore — it always does the same thing. A probabilistic policy sometimes tries the 'wrong' action. If that accident works out well, the model should increase the probability of that action. If it works out badly, decrease it. The randomness IS the learning signal.
</details>

<details><summary>Solution</summary>

A deterministic policy is stuck — it can't discover better actions. A probabilistic policy explores by sometimes taking random actions. When an accident leads to high reward, increase its probability. When it leads to low reward, decrease it. Exploration drives learning.


</details>

### Exercise 7.1 — A Probabilistic Policy

Convert your `weighted_policy` from a hard yes/no decision into a probability. Instead of "go right if `weights @ obs > 0`", output a PROBABILITY of going right.

You need a function that takes any number and squashes it to a value between 0 and 1. You've seen this before — what function does that?


In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

The sigmoid function! `sigmoid(weights @ obs)` gives a probability between 0 and 1. When `weights @ obs` is large and positive, sigmoid → 1 (almost certainly go right). When large and negative, sigmoid → 0 (almost certainly go left). When 0, sigmoid → 0.5 (coin flip).
</details>

<details><summary>Hint 2</summary>

To sample an action from the probability: generate a random number between 0 and 1. If it's less than the probability, go right (1). Otherwise go left (0).
</details>

<details><summary>Solution</summary>

```python
weights = np.zeros(4)

def prob_right(obs, weights):
    return sigmoid(np.dot(weights, obs))

def sample_action(obs, weights):
    p = prob_right(obs, weights)
    return 1 if np.random.random() < p else 0

# Test
obs, _ = env.reset()
p = prob_right(obs, weights)
print(f"Probability of going right: {p:.3f}")
print(f"Sampled action: {sample_action(obs, weights)}")
```

</details>

### Exercise 7.2 — How Should the Model Learn?

Your model outputs a probability and samples an action. After the game, you have discounted rewards telling you which actions were good (positive) and which were bad (negative).


**Your answer:** If action = 'go right' and the discounted reward was POSITIVE (good action), should the model make 'go right' MORE or LESS likely next time?

**Your answer:** If action = 'go right' and the discounted reward was NEGATIVE (bad action), what should happen?

**Your answer:** In gradient descent, you computed gradients and moved weights to DECREASE loss. Here you want to INCREASE reward. What direction should you move the weights?

**Your answer:** To compute a gradient, you need a single number that measures 'how likely was the action I chose.' The probability p does that — but probabilities multiply across steps, which is awkward for calculus. What mathematical function turns products into sums and is easy to differentiate?

<details><summary>Hint 1</summary>

Good action → make it more likely (increase probability). Bad action → make it less likely (decrease probability). To increase reward instead of decreasing loss, move in the gradient direction (ascent) instead of against it (descent). Gradient ASCENT.
</details>

<details><summary>Hint 2</summary>

The function is log! log(a × b) = log(a) + log(b). And log is smooth and differentiable. So you want the gradient of log(probability of chosen action) — that's your learning signal.
</details>

<details><summary>Solution</summary>

Good actions → increase their probability. Bad actions → decrease their probability. Use gradient ASCENT (move with the gradient) instead of gradient descent (move against it). Multiply the gradient by the reward: positive reward amplifies, negative reward reverses. Use log(p) as the quantity to differentiate — it turns products into sums and has clean gradients.


</details>

### Exercise 7.3 — Derive the Gradient

You just figured out that $\log(p)$ is the right quantity to differentiate. Now work out the gradient.

You already know from the Gradient Descent chapter that the derivative of sigmoid is $\text{sigmoid}(z) \cdot (1 - \text{sigmoid}(z))$, and that $\log'(x) = 1/x$.

Using the chain rule, work out: what is the gradient of $\log(\text{sigmoid}(\mathbf{w} \cdot \mathbf{obs}))$ with respect to $\mathbf{w}$?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Chain rule: $\frac{d}{dw} \log(\sigma(w \cdot obs))$ = $\frac{1}{\sigma} \cdot \sigma(1-\sigma) \cdot obs$ = $(1-\sigma) \cdot obs$ = $(1-p) \cdot obs$. The sigmoid derivative and the 1/sigmoid cancel beautifully!
</details>

<details><summary>Hint 2</summary>

That's the gradient when you went right (action=1). When you went left (action=0), you need $\log(1-p)$ instead. By similar math: gradient = $-p \cdot obs$.
</details>

<details><summary>Hint 3</summary>

Summary: action=1 (right) → gradient = $(1-p) \cdot obs$. Action=0 (left) → gradient = $-p \cdot obs$.
</details>

<details><summary>Solution</summary>

- Action = 1 (went right): gradient = $(1-p) \cdot \text{obs}$
- Action = 0 (went left): gradient = $-p \cdot \text{obs}$

This tells you which direction to push the weights to make the chosen action more likely.


</details>

### Exercise 7.4 — Play, Record, and Learn

Now put it all together. Write `play_episode(env, weights)` that:

1. Plays one full game using the probabilistic policy
2. At each step, records: observation, action, reward, and the gradient you derived above

Then write `reinforce_update(env, weights, lr)` that:

1. Plays an episode
2. Computes discounted rewards and normalizes them
3. Multiplies each step's gradient by its normalized reward
4. Sums all the weighted gradients
5. Updates: `weights += lr * total_gradient` (gradient ASCENT!)


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

For play_episode: at each step, compute `p = prob_right(obs, weights)`, sample an action, then compute the gradient: `(1-p)*obs` if action=1, `(-p)*obs` if action=0. Store everything in lists.
</details>

<details><summary>Hint 2</summary>

```python
def play_episode(env, weights):
    obs, _ = env.reset()
    observations, actions, rewards, gradients = [], [], [], []
    done = False
    while not done:
        p = prob_right(obs, weights)
        action = 1 if np.random.random() < p else 0
        grad = (1 - p) * obs if action == 1 else -p * obs
        observations.append(obs)
        actions.append(action)
        gradients.append(grad)
        obs, reward, terminated, truncated, _ = env.step(action)
        rewards.append(reward)
        done = terminated or truncated
    return observations, actions, rewards, gradients
```

</details>

<details><summary>Hint 3</summary>

```python
def reinforce_update(env, weights, lr=0.01):
    _, _, rewards, gradients = play_episode(env, weights)
    disc_rewards = compute_discounted_rewards(rewards)
    norm_rewards = normalize(disc_rewards)
    total_gradient = np.zeros(4)
    for grad, r in zip(gradients, norm_rewards):
        total_gradient += grad * r
    weights += lr * total_gradient
    return weights, sum(rewards)
```

</details>

<details><summary>Solution</summary>

```python
def play_episode(env, weights):
    obs, _ = env.reset()
    observations, actions, rewards, gradients = [], [], [], []
    done = False
    while not done:
        p = prob_right(obs, weights)
        action = 1 if np.random.random() < p else 0
        grad = (1 - p) * obs if action == 1 else -p * obs
        observations.append(obs)
        actions.append(action)
        gradients.append(grad)
        obs, reward, terminated, truncated, _ = env.step(action)
        rewards.append(reward)
        done = terminated or truncated
    return observations, actions, rewards, gradients

def reinforce_update(env, weights, lr=0.01):
    _, _, rewards, gradients = play_episode(env, weights)
    disc_rewards = compute_discounted_rewards(rewards)
    norm_rewards = normalize(disc_rewards)
    total_gradient = np.zeros(4)
    for grad, r in zip(gradients, norm_rewards):
        total_gradient += grad * r
    weights += lr * total_gradient
    return weights, sum(rewards)
```

</details>

Here's what a typical REINFORCE training curve looks like. Notice the high variance — scores bounce around wildly because each episode is different and the model is still exploring.

![Typical REINFORCE learning curve — noisy but trending upward](img/reinforce_curve.png)


### Exercise 7.5 — Train the Agent!

Run the REINFORCE update in a loop for 1000 episodes. Record the score each episode and plot it. Does the agent learn to balance the pole?


In [ ]:
import matplotlib.pyplot as plt

def plot_scores(scores, window=50, title="Training"):
    plt.figure(figsize=(10, 4))
    plt.plot(scores, alpha=0.3, label="episode score")
    avg = [np.mean(scores[max(0,i-window):i+1]) for i in range(len(scores))]
    plt.plot(avg, label=f"{window}-episode average")
    plt.xlabel("Episode")
    plt.ylabel("Score")
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Initialize `weights = np.zeros(4)`. Loop: `weights, score = reinforce_update(env, weights)`. Append score to a list. Print progress every 100 episodes.
</details>

<details><summary>Hint 2</summary>

```python
weights = np.zeros(4)
scores = []
for episode in range(1000):
    weights, score = reinforce_update(env, weights, lr=0.01)
    scores.append(score)
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode+1}: avg last 100 = {np.mean(scores[-100:]):.1f}")
plot_scores(scores, title="REINFORCE Training")
```

</details>

<details><summary>Solution</summary>

```python
weights = np.zeros(4)
scores = []
for episode in range(1000):
    weights, score = reinforce_update(env, weights, lr=0.01)
    scores.append(score)
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode+1}: avg last 100 = {np.mean(scores[-100:]):.1f}")

plot_scores(scores, title="REINFORCE Training")
print(f"\nFinal weights: {weights}")
print(f"Final 100-episode average: {np.mean(scores[-100:]):.1f}")
```

</details>

You just invented the **REINFORCE** algorithm (Williams, 1992)! Look at what happened:

- The model started knowing nothing (all weights = 0, coin-flip decisions)
- It played games, making random choices
- Good accidents (high reward) → gradient got amplified → model repeats those choices
- Bad accidents (low reward) → gradient got reversed → model avoids those choices
- Over hundreds of games, the model converged on a good policy

No one told the model the rules of physics. It learned entirely from its own experience — just like you did in Part 1, but automatically.


## Part 8: Predicting Value

REINFORCE works, but notice something: it has to finish the ENTIRE game before it can learn anything. It plays 200 steps, then looks back and assigns credit. Can we do better?


### Exercise 8.1 — You Can Already Predict

Think about two situations mid-game:


**Your answer:** The pole is nearly vertical, barely moving. How many more steps do you think you'll survive? What's your expected future reward from here?

**Your answer:** The pole is nearly horizontal, falling fast. Expected future reward?

**Your answer:** You just made an intuitive prediction about future reward based on the state. If a MODEL could make this prediction, why would that be useful for learning?

<details><summary>Hint 1</summary>

You could learn after EVERY step instead of waiting until the end! Take action → see next state → model predicts future reward for next state. If the predicted future is better than you expected, the action was good. If worse, the action was bad. Instant feedback!
</details>

<details><summary>Solution</summary>

If you can predict future reward from any state, you get instant feedback. After each action, compare the predicted value of the next state to what you expected. No need to wait for the game to end.


</details>

### Exercise 8.2 — Build a Value Predictor

Build a simple linear model: given a state (observation), predict the total future reward.

$$V(\text{obs}) = \mathbf{v} \cdot \text{obs}$$

To train it, you need data: play episodes using your REINFORCE policy, compute actual discounted rewards at each step (you already have `compute_discounted_rewards`!), and train the model to predict those rewards using the MSE loss from the Gradient Descent chapter.


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

This is linear regression from the Gradient Descent chapter! Input = observation (4 numbers). Target = actual discounted return at that step. Loss = $(V(obs) - target)^2$. Gradient = $2 \cdot (V(obs) - target) \cdot obs$.
</details>

<details><summary>Hint 2</summary>

```python
value_weights = np.zeros(4)

def predict_value(obs, value_weights):
    return np.dot(value_weights, obs)
```

</details>

<details><summary>Hint 3</summary>

Train it by playing episodes and fitting to discounted returns:
```python
def train_value(observations, disc_rewards, value_weights, lr=0.001):
    for obs, target in zip(observations, disc_rewards):
        pred = predict_value(obs, value_weights)
        error = pred - target
        value_weights -= lr * 2 * error * obs
    return value_weights
```

</details>

<details><summary>Solution</summary>

```python
value_weights = np.zeros(4)

def predict_value(obs, value_weights):
    return np.dot(value_weights, obs)

def train_value(observations, disc_rewards, value_weights, lr=0.001):
    for obs, target in zip(observations, disc_rewards):
        pred = predict_value(obs, value_weights)
        error = pred - target
        value_weights -= lr * 2 * error * obs
    return value_weights
```

</details>

### Exercise 8.3 — The Waiting Problem

Your value model works — but it still needs `compute_discounted_rewards`, which requires the FULL list of rewards from a completed episode.


**Your answer:** Can you think of a way to estimate the discounted return at step t WITHOUT finishing the game?

**Your answer:** You have: the reward at step t, and the value model's prediction for the NEXT state. The discounted return at step t is: reward[t] + gamma × (discounted return at step t+1). But the discounted return at step t+1 is approximately... what?

**Your answer:** So the discounted return at step t is approximately: reward[t] + gamma × V(next_state). Does this require finishing the game?

<details><summary>Hint 1</summary>

The value model already predicts the discounted return! So instead of waiting, use: `estimated_return = reward + gamma * V(next_state)`. This only needs the current step's reward and the next state. No waiting!
</details>

<details><summary>Solution</summary>

Use the value model itself as the estimate: `target = reward + gamma × V(next_state)`. You only need the current reward and the next state — both available immediately. No need to finish the episode!


</details>

### Exercise 8.4 — Step-by-Step Value Learning

Implement the idea you just derived. At each step, compute:

- `target = reward + gamma * V(next_state)` (your estimate of the true return)
- `error = target - V(current_state)` (how wrong the value model was)
- Update: `value_weights += lr * error * obs`

Write `td_update(obs, reward, next_obs, done, value_weights)`. When the game is done, `V(next_state) = 0` (no future rewards).


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

When `done` is True, set `next_v = 0` since there's no future. Otherwise `next_v = predict_value(next_obs, value_weights)`.
</details>

<details><summary>Hint 2</summary>

The error `target - V(current)` tells you something interesting: was the actual outcome BETTER or WORSE than what the model predicted? Positive error = better than expected. Negative = worse.
</details>

<details><summary>Hint 3</summary>

```python
def td_update(obs, reward, next_obs, done, value_weights, gamma=0.99, lr=0.001):
    current_v = predict_value(obs, value_weights)
    next_v = 0 if done else predict_value(next_obs, value_weights)
    target = reward + gamma * next_v
    error = target - current_v
    value_weights += lr * error * obs
    return value_weights, error
```

</details>

<details><summary>Solution</summary>

```python
def td_update(obs, reward, next_obs, done, value_weights, gamma=0.99, lr=0.001):
    current_v = predict_value(obs, value_weights)
    next_v = 0.0 if done else predict_value(next_obs, value_weights)
    target = reward + gamma * next_v
    error = target - current_v
    value_weights += lr * error * obs
    return value_weights, error
```

</details>

### Exercise 8.5 — What Does the Error Mean?

Look at the error from `td_update`: `error = reward + gamma * V(next_state) - V(current_state)`.


**Your answer:** If the error is positive, it means: what you actually got (reward + predicted future) is MORE than what you expected (V(current)). Was the action that got you here good or bad?

**Your answer:** If the error is negative, was the action good or bad?

**Your answer:** Compare this error to the normalized discounted rewards from REINFORCE. Which is a better signal for telling the actor which actions were good?

<details><summary>Hint 1</summary>

Positive error = action was BETTER than expected. Negative = WORSE than expected. This 'better or worse than expected' signal is much less noisy than raw rewards, because it's relative to a learned baseline (the value model's prediction), not just the episode average.
</details>

<details><summary>Solution</summary>

The error measures **advantage** — was the outcome better or worse than expected? Positive = good action, negative = bad action. It's less noisy than raw discounted rewards because the value model provides a per-state baseline instead of the crude episode-wide average.


</details>

### Exercise 8.6 — Compare the Signals

Play an episode and compute both: (1) the normalized discounted rewards from REINFORCE, and (2) the TD errors (advantages) from the value model. Compare their standard deviations. Which signal is more stable?


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

Play an episode with `play_episode`. Get discounted rewards and normalize them. Then replay the same states computing TD errors. Print the std of each.
</details>

<details><summary>Hint 2</summary>

```python
obs, _ = env.reset()
advantages = []
rewards_list = []
done = False
while not done:
    action = sample_action(obs, weights)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    v_current = predict_value(obs, value_weights)
    v_next = 0.0 if done else predict_value(next_obs, value_weights)
    advantage = reward + 0.99 * v_next - v_current
    advantages.append(advantage)
    rewards_list.append(reward)
    obs = next_obs

norm_disc = normalize(compute_discounted_rewards(rewards_list))
print(f"Advantage std: {np.std(advantages):.4f}")
print(f"Norm discounted reward std: {np.std(norm_disc):.4f}")
```

</details>

<details><summary>Solution</summary>

```python
obs, _ = env.reset()
advantages = []
rewards_list = []
done = False
while not done:
    action = sample_action(obs, weights)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    v_current = predict_value(obs, value_weights)
    v_next = 0.0 if done else predict_value(next_obs, value_weights)
    advantage = reward + 0.99 * v_next - v_current
    advantages.append(advantage)
    rewards_list.append(reward)
    obs = next_obs

norm_disc = normalize(compute_discounted_rewards(rewards_list))
print(f"Advantage std: {np.std(advantages):.4f}")
print(f"Norm discounted reward std: {np.std(norm_disc):.4f}")
```

</details>

You just invented several things at once:

- A model that predicts future reward from a state = the **value function** (or **critic**)
- Training it step-by-step using `reward + gamma × V(next_state)` = **Temporal Difference (TD) learning**
- The error `reward + gamma × V(next) - V(current)` = the **advantage** (or **TD error**)
- With a neural network instead of linear weights = **Deep Q-Network (DQN)**

The value model doesn't pick actions — it just evaluates states. That's why it's called the **critic**: it watches the policy play and judges how well it's doing.


## Part 9: Best of Both Worlds — Actor-Critic

You now have two models:
1. The **actor** (REINFORCE policy from Part 7) — picks actions
2. The **critic** (value model from Part 8) — evaluates states

What if you used BOTH together? The critic tells the actor how good its actions are (via advantages), and the actor uses that feedback to improve. The critic learns alongside the actor, getting better at evaluating states as the actor gets better at playing.


### Exercise 9.1 — Two Models Working Together

Think about how to combine the actor and critic:


**Your answer:** In REINFORCE, you multiplied gradients by normalized discounted rewards. But you now have a better signal — the advantage (TD error). What should you multiply the gradients by instead?

**Your answer:** Both models learn at the same time. What does the actor learn from? What does the critic learn from?

**Your answer:** REINFORCE waits until the end of the episode. With a critic providing advantages at each step, do you still need to wait?

<details><summary>Hint 1</summary>

The actor uses advantages instead of discounted rewards — gradient × advantage. The critic learns to predict value (minimizes TD error). And no, you don't need to wait — you can update BOTH models at every single step!
</details>

<details><summary>Solution</summary>

Actor update: gradient × advantage (instead of gradient × discounted_reward). Critic update: minimize TD error (same as before). Both update every step — no need to finish the episode. The critic provides instant feedback to the actor, and gets better at predicting as the actor improves.


</details>

Here's a typical Actor-Critic learning curve. Compare it to the REINFORCE curve — notice how Actor-Critic converges faster and with less variance, because the critic provides a per-step learning signal instead of waiting until the episode ends.

![Typical Actor-Critic learning curve — faster convergence, less variance](img/actor_critic_curve.png)


### Exercise 9.2 — Build Actor-Critic

Combine the actor and critic into a single training loop. At each step within an episode:

1. Actor picks action (probabilistic, using sigmoid)
2. Observe reward and next state
3. Critic computes advantage (TD error)
4. Update actor weights using advantage × gradient
5. Update critic weights using TD error

Train for 2000 episodes. Plot the scores.


In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

You already have all the pieces. The actor is `prob_right` + `sample_action` from Part 7. The critic is `predict_value` + TD error from Part 8. The new part is doing both updates at each step.
</details>

<details><summary>Hint 2</summary>

Actor gradient: `(1-p)*obs` if action=1, `(-p)*obs` if action=0. Actor update: `actor_weights += actor_lr * td_error * grad`. Critic update: `critic_weights += critic_lr * td_error * obs`.
</details>

<details><summary>Hint 3</summary>

```python
actor_weights = np.zeros(4)
critic_weights = np.zeros(4)
actor_lr = 0.01
critic_lr = 0.005
gamma = 0.99
scores = []

for episode in range(2000):
    obs, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        p = prob_right(obs, actor_weights)
        action = 1 if np.random.random() < p else 0
        grad = (1 - p) * obs if action == 1 else -p * obs

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        v_current = predict_value(obs, critic_weights)
        v_next = 0 if done else predict_value(next_obs, critic_weights)
        td_error = reward + gamma * v_next - v_current

        actor_weights += actor_lr * td_error * grad
        critic_weights += critic_lr * td_error * obs
        obs = next_obs

    scores.append(total_reward)
    if (episode + 1) % 200 == 0:
        print(f"Episode {episode+1}: avg = {np.mean(scores[-200:]):.1f}")
```

</details>

<details><summary>Solution</summary>

```python
actor_weights = np.zeros(4)
critic_weights = np.zeros(4)
actor_lr = 0.01
critic_lr = 0.005
gamma = 0.99
scores = []

for episode in range(2000):
    obs, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        p = prob_right(obs, actor_weights)
        action = 1 if np.random.random() < p else 0
        grad = (1 - p) * obs if action == 1 else -p * obs

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        v_current = predict_value(obs, critic_weights)
        v_next = 0.0 if done else predict_value(next_obs, critic_weights)
        td_error = reward + gamma * v_next - v_current

        actor_weights += actor_lr * td_error * grad
        critic_weights += critic_lr * td_error * obs

        obs = next_obs

    scores.append(total_reward)
    if (episode + 1) % 200 == 0:
        print(f"Episode {episode+1}: avg last 200 = {np.mean(scores[-200:]):.1f}")

plot_scores(scores, title="Actor-Critic Training")
```

</details>

### Exercise 9.3 — The Grand Comparison

You've built four approaches. Let's see them side by side. Run each one and plot their learning curves together:

1. **Random** — baseline (random actions)
2. **Genetic Algorithm** — 10 generations of evolution
3. **REINFORCE** — 1000 episodes of policy gradient
4. **Actor-Critic** — 2000 episodes of combined learning

Which learns fastest? Which achieves the highest score?


In [ ]:
def plot_comparison(results):
    plt.figure(figsize=(12, 5))
    for name, scores in results.items():
        avg = [np.mean(scores[max(0,i-50):i+1]) for i in range(len(scores))]
        plt.plot(avg, label=name)
    plt.xlabel("Episode")
    plt.ylabel("50-episode average score")
    plt.title("Comparison of RL Approaches")
    plt.legend()
    plt.show()

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

For the genetic algorithm, you'll need to adapt it to record a score per 'episode' — you can record the best score each generation, or evaluate the current best weights each episode.
</details>

<details><summary>Hint 2</summary>

Collect scores as `{'Random': [...], 'REINFORCE': [...], 'Actor-Critic': [...]}` and pass to `plot_comparison`.
</details>

<details><summary>Solution</summary>

```python
# Random baseline
random_scores = []
for _ in range(1000):
    random_scores.append(play_game(lambda obs: np.random.choice([0, 1]), env))

# REINFORCE
w_reinforce = np.zeros(4)
reinforce_scores = []
for _ in range(1000):
    w_reinforce, score = reinforce_update(env, w_reinforce, lr=0.01)
    reinforce_scores.append(score)

# Actor-Critic
aw = np.zeros(4)
cw = np.zeros(4)
ac_scores = []
for _ in range(1000):
    obs, _ = env.reset()
    total = 0
    done = False
    while not done:
        p = prob_right(obs, aw)
        action = 1 if np.random.random() < p else 0
        grad = (1 - p) * obs if action == 1 else -p * obs
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total += reward
        v_cur = predict_value(obs, cw)
        v_nxt = 0.0 if done else predict_value(next_obs, cw)
        td_err = reward + 0.99 * v_nxt - v_cur
        aw += 0.01 * td_err * grad
        cw += 0.005 * td_err * obs
        obs = next_obs
    ac_scores.append(total)

plot_comparison({'Random': random_scores, 'REINFORCE': reinforce_scores, 'Actor-Critic': ac_scores})
```

</details>

### Go Deeper — Neural Network Actor-Critic

Replace the linear models with small neural networks. Use one hidden layer of 32 neurons with ReLU activation. You can use numpy (manual backprop) or PyTorch/TensorFlow.

Does a deeper model learn faster or achieve a higher score than the linear version?


In [ ]:
class SimpleNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.w1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros(hidden_size)
        self.w2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros(output_size)

    def forward(self, x):
        self.z1 = x @ self.w1 + self.b1
        self.h = np.maximum(0, self.z1)  # ReLU
        self.z2 = self.h @ self.w2 + self.b2
        return self.z2

    # You'll need to implement backprop for this!

In [ ]:
# YOUR CODE HERE


<details><summary>Hint 1</summary>

If using PyTorch, this becomes much simpler — `nn.Linear(4, 32)`, `nn.ReLU()`, `nn.Linear(32, 1)`, and `loss.backward()` handles all the gradient computation.
</details>

<details><summary>Hint 2</summary>

With numpy, you'll need the ReLU derivative: `d_relu = (self.z1 > 0).astype(float)`. Chain rule for w2: `d_w2 = self.h.T @ d_output`. For w1: `d_w1 = x.T @ (d_output @ self.w2.T * d_relu)`.
</details>

## Summary

Look at how far you came:

1. You played CartPole with random actions (scored ~20)
2. You hand-coded a heuristic (scored ~60)
3. You evolved weights with a genetic algorithm (scored ~200+)
4. You tried gradient estimation (learned why it's noisy)
5. You invented discounted rewards (credit assignment)
6. You invented REINFORCE (policy gradient)
7. You invented a value function (the critic)
8. You combined them into Actor-Critic

Every single one of these is a real algorithm used in production. The only difference between yours and DeepMind's is scale — they use massive neural networks, millions of episodes, and GPU clusters. The ideas are identical.


### What You Built vs. What the Field Calls It

| What you built | The field calls it |
| --- | --- |
| Hand-coded if-else rules | Heuristic policy |
| `weighted_policy(obs, weights)` | Linear policy / parameterized policy |
| Try 100 random weights, keep the best | Random search |
| Keep top 20, mutate, repeat | Genetic algorithm / evolutionary strategy |
| Tweak weight, measure score change | Finite-difference policy gradient |
| `compute_discounted_rewards()` | Discounted return / reward-to-go |
| Multiply gradient by reward, update | REINFORCE (Williams, 1992) |
| Model that predicts future reward | Value function / critic |
| $r + \gamma V(s') - V(s)$ | TD error / advantage |
| Step-by-step value updates | Temporal Difference (TD) learning |
| Actor picks actions, critic evaluates | Actor-Critic |
| Neural net instead of linear | Deep RL (DQN, A2C, PPO) |